# Cross-attention
1. Cross-attention by hand on "we're friends ." -> "nous sommes amis": queries from the French side,
   keys and values from the English side; checked against keras.layers.MultiHeadAttention.
2. Shapes: self-attention gives an m x m table, cross-attention an m x n table; one output per French word.
3. A small English -> French transformer trained on the Keras English-French corpus;
   its cross-attention weights on two sentence pairs it never saw.

In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import re, time
from pathlib import Path
import numpy as np
import pandas as pd
import keras
from keras import ops
import tensorflow as tf

tf.config.experimental.enable_tensor_float_32_execution(False)   # full float32 precision on the GPU

np.set_printoptions(precision=3, suppress=True)
Path("data").mkdir(exist_ok=True)


def softmax_rows(S):
    W = np.exp(S - S.max(-1, keepdims=True))
    return W / W.sum(-1, keepdims=True)

I0000 00:00:1791081751.874247 2084334 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


## 1. Cross-attention by hand
English (encoder side, n = 3 words): "we're friends ." Its encoder output H_enc is a stand-in: random numbers,
8 per word. French (decoder side, m = 4 positions): "<start> nous sommes amis", also 8 random numbers per word.

In [2]:
keras.utils.set_random_seed(0)                # same random W_Q, W_K, W_V on every run
rng = np.random.default_rng(0)
d = 8
eng = ["we're", "friends", "."]
fra = ["<start>", "nous", "sommes", "amis"]
H_enc = rng.normal(size=(len(eng), d)).round(2)
X_dec = rng.normal(size=(len(fra), d)).round(2)
mha = keras.layers.MultiHeadAttention(num_heads=1, key_dim=d, use_bias=False)
mha(X_dec[None].astype("float32"), H_enc[None].astype("float32"))          # builds the layer
Wq, Wk, Wv, Wo = mha.get_weights()
mha.set_weights([Wq, Wk, Wv, np.eye(d)[None]])                           # output matrix = identity
Wq, Wk, Wv = Wq[:, 0], Wk[:, 0], Wv[:, 0]

Q = X_dec @ Wq          # queries from the French side   (4 x 8)
K = H_enc @ Wk          # keys from the English side     (3 x 8)
V = H_enc @ Wv          # values from the English side   (3 x 8)
S = Q @ K.T / np.sqrt(d)
A = softmax_rows(S)
Y = A @ V
print("Q", Q.shape, " K", K.shape, " V", V.shape, " scores", S.shape, " weights", A.shape, " output", Y.shape)
print("scaled scores:\n", S)
print("weights (rows: French positions, columns: English words):\n", A)
print("row sums:", A.sum(1))
out_k, w_k = mha(X_dec[None].astype("float32"), H_enc[None].astype("float32"), H_enc[None].astype("float32"),
                 return_attention_scores=True)
print("largest difference to Keras, weights:", np.abs(w_k.numpy()[0, 0] - A).max(), " outputs:", np.abs(out_k.numpy()[0] - Y).max())
assert np.allclose(w_k.numpy()[0, 0], A, atol=1e-5) and np.allclose(out_k.numpy()[0], Y, atol=1e-5)
pd.DataFrame(A, index=fra, columns=eng).round(3).to_csv("data/by_hand_weights.csv")

Q (4, 8)  K (3, 8)  V (3, 8)  scores (4, 3)  weights (4, 3)  output (4, 8)
scaled scores:
 [[ 0.34  -0.349 -0.143]
 [ 0.016  0.505 -0.145]
 [-0.008  1.035 -0.15 ]
 [ 0.492  0.3   -0.125]]
weights (rows: French positions, columns: English words):
 [[0.472 0.237 0.291]
 [0.287 0.468 0.244]
 [0.213 0.603 0.184]
 [0.423 0.349 0.228]]
row sums: [1. 1. 1. 1.]
largest difference to Keras, weights: 3.7916219497446946e-08  outputs: 8.683004293352781e-08


The same layer used as self-attention on the French side gives a 4 x 4 table; as cross-attention, 4 x 3.

In [3]:
_, w_self = mha(X_dec[None].astype("float32"), X_dec[None].astype("float32"), return_attention_scores=True)
print("self-attention weights:", tuple(w_self.shape[2:]), "  cross-attention weights:", tuple(w_k.shape[2:]))

self-attention weights: (4, 4)   cross-attention weights: (4, 3)


A longer English input changes the number of columns, never the number of outputs.

In [4]:
H_long = rng.normal(size=(7, d)).astype("float32")
out_long, w_long = mha(X_dec[None].astype("float32"), H_long[None], return_attention_scores=True)
print("7 English words: weights", tuple(w_long.shape[2:]), " outputs", tuple(out_long.shape[1:]))

7 English words: weights (4, 7)  outputs (4, 8)


Parameters: a cross-attention layer at the paper's sizes has the same weights as a self-attention layer.

In [5]:
for name, kv_len in (("self-attention", 10), ("cross-attention", 7)):
    layer = keras.layers.MultiHeadAttention(num_heads=8, key_dim=64)
    layer(np.zeros((1, 10, 512), "float32"), np.zeros((1, kv_len, 512), "float32"))
    print(f"{name}: {layer.count_params():,} parameters")

self-attention: 1,050,624 parameters


cross-attention: 1,050,624 parameters


## 2. A small trained English -> French transformer
Data: the English-French sentence pairs of the Keras examples (Tatoeba). Lower case, punctuation split off,
up to 10 tokens per side. The two English sentences used for the pictures are removed from training.

In [6]:
path = keras.utils.get_file("fra-eng.zip", origin="http://storage.googleapis.com/download.tensorflow.org/data/fra-eng.zip",
                            extract=True)
lines = Path(path, "fra.txt").read_text(encoding="utf-8").strip().split("\n")
tok = lambda s: re.findall(r"[\w']+|[^\w\s]", s.lower())
SHOW = ["Tom likes to eat ice cream.", "I want ice cream for dessert."]
pairs = [(tok(e), tok(f)) for e, f in (l.split("\t")[:2] for l in lines) if e not in SHOW]
pairs = [(e, f) for e, f in pairs if len(e) <= 10 and len(f) <= 10]
rng_split = np.random.default_rng(1)
rng_split.shuffle(pairs)
print(len(pairs), "pairs, e.g.", pairs[0])


def vocab(seqs, size):
    words = pd.Series([w for s in seqs for w in s]).value_counts().index[:size - 4]
    return ["<pad>", "<unk>", "<start>", "<end>"] + list(words)


ev, fv = vocab([e for e, _ in pairs], 8000), vocab([f for _, f in pairs], 10000)
ei, fi = {w: i for i, w in enumerate(ev)}, {w: i for i, w in enumerate(fv)}
N_IN, N_OUT = 10, 11                                     # French side: <start> + 10 words, or 10 words + <end>


def encode(e, f):
    x = [ei.get(w, 1) for w in e] + [0] * (N_IN - len(e))
    y = [fi.get(w, 1) for w in f]
    din = [2] + y + [0] * (N_OUT - 1 - len(y))
    dout = y + [3] + [0] * (N_OUT - 1 - len(y))
    return x, din, dout


arr = np.array([sum(encode(e, f), []) for e, f in pairs], dtype="int32")
X, DIN, DOUT = arr[:, :N_IN], arr[:, N_IN:N_IN + N_OUT], arr[:, N_IN + N_OUT:]
n_val = 5000
print("train", len(X) - n_val, " validation", n_val, " vocab", len(ev), len(fv))

136463 pairs, e.g. (['i', "can't", 'eat', 'pork', '.'], ['je', "n'arrive", 'pas', 'à', 'manger', 'du', 'porc', '.'])


train 131463  validation 5000  vocab 8000 10000


The model: embeddings + sinusoidal positional encodings, one encoder block, one decoder block
(masked self-attention, cross-attention, feed-forward network), d_model = 128, 4 heads.
The decoder block itself is the subject of the decoder Note.

In [7]:
D, H, FF = 128, 4, 512


def pos_enc(n, d):
    p, i = np.arange(n)[:, None], np.arange(0, d, 2)[None]
    pe = np.zeros((n, d), "float32")
    pe[:, 0::2], pe[:, 1::2] = np.sin(p / 10000 ** (i / d)), np.cos(p / 10000 ** (i / d))
    return pe


class Seq2SeqTransformer(keras.Model):
    def __init__(self):
        super().__init__()
        self.emb_e, self.emb_f = keras.layers.Embedding(len(ev), D), keras.layers.Embedding(len(fv), D)
        self.pe_e, self.pe_f = pos_enc(N_IN, D), pos_enc(N_OUT, D)
        self.enc_att = keras.layers.MultiHeadAttention(H, D // H)
        self.dec_self = keras.layers.MultiHeadAttention(H, D // H)
        self.cross = keras.layers.MultiHeadAttention(H, D // H)
        self.ffe = keras.Sequential([keras.layers.Dense(FF, activation="relu"), keras.layers.Dense(D)])
        self.ffd = keras.Sequential([keras.layers.Dense(FF, activation="relu"), keras.layers.Dense(D)])
        self.norms = [keras.layers.LayerNormalization() for _ in range(5)]
        self.drop = keras.layers.Dropout(0.1)
        self.out = keras.layers.Dense(len(fv))

    def call(self, inputs, training=False, return_cross=False):
        x_ids, y_ids = inputs
        xm, ym = ops.not_equal(x_ids, 0), ops.not_equal(y_ids, 0)
        x = self.emb_e(x_ids) * np.sqrt(D) + self.pe_e
        enc_mask = ops.logical_and(xm[:, None, :], xm[:, :, None])
        x = self.norms[0](x + self.drop(self.enc_att(x, x, attention_mask=enc_mask), training=training))
        h_enc = self.norms[1](x + self.drop(self.ffe(x), training=training))
        y = self.emb_f(y_ids) * np.sqrt(D) + self.pe_f
        y = self.norms[2](y + self.drop(self.dec_self(y, y, attention_mask=ym[:, None, :], use_causal_mask=True),
                                        training=training))
        c, w = self.cross(y, h_enc, h_enc, attention_mask=ops.logical_and(ym[:, :, None], xm[:, None, :]),
                          return_attention_scores=True)
        y = self.norms[3](y + self.drop(c, training=training))
        y = self.norms[4](y + self.drop(self.ffd(y), training=training))
        logits = self.out(y)
        return (logits, w) if return_cross else logits


keras.utils.set_random_seed(0)
model = Seq2SeqTransformer()
model.compile(optimizer=keras.optimizers.Adam(5e-4),
              loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True, ignore_class=0))
t0 = time.time()
hist = model.fit((X[n_val:], DIN[n_val:]), DOUT[n_val:], validation_data=((X[:n_val], DIN[:n_val]), DOUT[:n_val]),
                 epochs=6, batch_size=128, verbose=2)
print(f"training time {time.time() - t0:.0f} s, parameters {model.count_params():,}")
pd.DataFrame(hist.history).rename_axis("epoch").round(4).to_csv("data/history.csv")

Epoch 1/6


1028/1028 - 65s - 63ms/step - loss: 3.0855 - val_loss: 1.7540


Epoch 2/6


1028/1028 - 40s - 39ms/step - loss: 1.4204 - val_loss: 1.2352


Epoch 3/6


...

Epoch 5/6


1028/1028 - 40s - 39ms/step - loss: 0.6835 - val_loss: 0.9576


Epoch 6/6


1028/1028 - 36s - 35ms/step - loss: 0.6042 - val_loss: 0.9593


training time 262 s, parameters 4,056,848


Next-word accuracy on the 5,000 validation pairs (padding positions excluded), with the correct previous words given.

In [8]:
pred = np.concatenate([model.predict((X[i:i + 1000], DIN[i:i + 1000]), verbose=0).argmax(-1) for i in range(0, n_val, 1000)])
keep = DOUT[:n_val] != 0
print("validation next-word accuracy:", round(float((pred[keep] == DOUT[:n_val][keep]).mean()), 3))

validation next-word accuracy: 0.784


## 3. Cross-attention weights of the trained model on the two held-out pairs
The decoder is given the reference French translation (as in training) and we read the weights of the
cross-attention layer, averaged over its 4 heads. Rows: French positions, labelled "input word -> word it predicts";
columns: English words.

In [9]:
SHOW_FR = ["Tom aime bien manger de la glace.", "Je veux de la crème glacée pour le dessert."]
rows = []
for k, (e, f) in enumerate(zip(SHOW, SHOW_FR)):
    x, din, _ = encode(tok(e), tok(f))
    _, w = model((np.array([x]), np.array([din])), return_cross=True)
    w = w.numpy()[0]                                         # heads x 11 x 10
    n, m = len(tok(e)), len(tok(f)) + 1
    avg = w.mean(0)[:m, :n]
    print(e, "->", f)
    targets = tok(f) + ["<end>"]
    print(pd.DataFrame(avg, index=[f"{a} -> {b}" for a, b in zip(["<start>"] + tok(f), targets)], columns=tok(e)).round(2), "\n")
    for qi, qw in enumerate(["<start>"] + tok(f)):
        for ki, kw in enumerate(tok(e)):
            rows.append(dict(pair=k, query_pos=qi, query=qw, predicts=targets[qi], key_pos=ki, key=kw, weight=round(float(avg[qi, ki]), 4),
                             **{f"head{h + 1}": round(float(w[h, qi, ki]), 4) for h in range(H)}))
    assert np.allclose(w[:, :m, :n].sum(-1), 1, atol=1e-4)        # every row of every head sums to 1
pd.DataFrame(rows).to_csv("data/cross_weights.csv", index=False)

Tom likes to eat ice cream. -> Tom aime bien manger de la glace.
                 tom  likes    to   eat   ice  cream     .
<start> -> tom  0.91   0.09  0.00  0.00  0.00   0.00  0.00
tom -> aime     0.01   0.88  0.04  0.05  0.00   0.01  0.00
aime -> bien    0.03   0.06  0.20  0.42  0.16   0.14  0.01
bien -> manger  0.04   0.02  0.13  0.36  0.16   0.27  0.01
manger -> de    0.01   0.00  0.02  0.05  0.34   0.40  0.18
de -> la        0.00   0.00  0.00  0.06  0.40   0.50  0.03
la -> glace     0.00   0.00  0.00  0.01  0.50   0.48  0.01
glace -> .      0.00   0.00  0.00  0.02  0.01   0.24  0.74
. -> <end>      0.00   0.00  0.02  0.03  0.17   0.08  0.69 

I want ice cream for dessert. -> Je veux de la crème glacée pour le dessert.
                    i  want   ice  cream   for  dessert     .
<start> -> je    0.69  0.28  0.02   0.01  0.00     0.01  0.00
je -> veux       0.02  0.92  0.05   0.01  0.00     0.00  0.00
veux -> de       0.01  0.02  0.63   0.30  0.01     0.03  0.00
de -> la         0

Where does each French word put most of its weight?

In [10]:
cw = pd.DataFrame(rows)
for k in (0, 1):
    t = cw[cw.pair == k].sort_values("weight").groupby("query_pos").tail(1).sort_values("query_pos")
    print(" | ".join(f"{q} -> {kk} ({w:.2f})" for q, kk, w in zip(t["predicts"], t["key"], t["weight"])))

tom -> tom (0.91) | aime -> likes (0.88) | bien -> eat (0.42) | manger -> eat (0.36) | de -> cream (0.40) | la -> cream (0.50) | glace -> ice (0.50) | . -> . (0.74) | <end> -> . (0.69)
je -> i (0.69) | veux -> want (0.92) | de -> ice (0.63) | la -> ice (0.60) | crème -> ice (0.65) | glacée -> cream (0.32) | pour -> . (0.63) | le -> dessert (0.69) | dessert -> dessert (0.72) | . -> . (0.78) | <end> -> . (0.71)
